# BTTH3 — Bài tập về nhà: Digital Health Records DApp (DHR-DApp)

Ứng dụng phi tập trung quản lý **hồ sơ sức khỏe điện tử**: bệnh nhân sở hữu dữ liệu, cấp/thu hồi quyền cho bác sĩ; mọi thao tác được ghi lại trên blockchain.

**Mã nguồn:** thư mục `dhr_dapp/` (cùng cấp notebook).

| Thành phần | File |
|---|---|
| Smart contract | `dhr_dapp/contracts/DHR.vy` |
| Unit test | `dhr_dapp/tests/conftest.py`, `dhr_dapp/tests/test_dhr.py` |
| Tạo / nạp tiền tài khoản | `dhr_dapp/scripts/create_accounts.py`, `dhr_dapp/scripts/fund_accounts.py` |
| Deploy | `dhr_dapp/scripts/deploy.py` |
| Kịch bản kiểm thử trên Geth | `dhr_dapp/scripts/demo_flow.py` |
| Giao diện | `dhr_dapp/dhr_gui_app.py` |

Môi trường: Windows, `.venv-ape` (Python 3.12, Ape 0.8.52, Vyper 0.4.3, PySide6), Geth 1.17.7 `--dev`.


## A. Mô tả bài toán

Hồ sơ y tế truyền thống nằm trên máy chủ tập trung của bệnh viện: bệnh nhân không kiểm soát được ai xem/sửa hồ sơ của mình, và dữ liệu có thể bị sửa mà không để lại dấu vết.

DHR-DApp đưa hồ sơ và quyền truy cập lên smart contract:

- **Bệnh nhân** là chủ sở hữu hồ sơ, tự quyết định bác sĩ nào được đọc/ghi và thu hồi bất kỳ lúc nào.
- **Bác sĩ** chỉ thao tác được khi đã được **bệnh viện xác thực** và được **bệnh nhân cấp quyền**.
- Mọi thao tác (tạo hồ sơ, cấp/thu hồi quyền, ghi, xem) được ghi thành **log on-chain** kèm địa chỉ ví và thời gian, không thể sửa.

## B. Đặc tả chức năng

| Đối tượng | Chức năng | Cài đặt trong contract |
|---|---|---|
| Bệnh nhân (Patient) | Tạo hồ sơ; cấp / thu hồi quyền cho bác sĩ; xem lịch sử truy cập | `registerPatient`, `grantAccess`, `revokeAccess`, `getLog` |
| Bác sĩ (Doctor) | Xem hồ sơ được cấp quyền; cập nhật chẩn đoán, đơn thuốc | `registerDoctor`, `viewHealthRecord`, `addHealthRecord`, `logView` |
| Cơ sở y tế (Hospital) | Xác thực danh tính bác sĩ | `verifyDoctor` (chỉ địa chỉ deploy contract) |
| Smart contract | Lưu hồ sơ, quản lý quyền (access control), ghi log | `records`, `permissions`, `logs` + các event |

## C. Cấu trúc hệ thống (3 tầng)

1. **Smart Contract (Blockchain layer):** `DHR.vy` viết bằng Vyper, triển khai bằng Ape lên Geth.
2. **Backend Script (Interaction layer):** các script Ape (`deploy.py`, `demo_flow.py`, …) gọi hàm contract, gửi giao dịch và in log.
3. **Frontend GUI (User layer):** `dhr_gui_app.py` (PySide6) gồm 5 tab: Đăng nhập, Hồ sơ sức khỏe, Cấp quyền truy cập, Lịch sử truy cập, Trạng thái giao dịch.


## D. Thiết kế smart contract `DHR.vy`

### Cấu trúc dữ liệu

| Struct | Trường | Ý nghĩa |
|---|---|---|
| `PatientRecord` | `name`, `birthYear`, `recordCount`, `logCount`, `exists` | Thông tin bệnh nhân + bộ đếm hồ sơ / log |
| `DoctorInfo` | `name`, `specialty`, `verified`, `exists` | Thông tin bác sĩ, `verified` do bệnh viện bật |
| `AccessPermission` | `canRead`, `canWrite`, `grantedAt` | Quyền của 1 bác sĩ trên hồ sơ 1 bệnh nhân |
| `HealthRecord` | `diagnosis`, `prescription`, `visitDate`, `note`, `doctor`, `timestamp` | Một lần khám |
| `ActivityLog` | `actor`, `action`, `target`, `timestamp` | Một dòng lịch sử truy cập |

Lưu trữ: `permissions[patient][doctor]`, `records[patient][i]`, `logs[patient][i]` (HashMap lồng nhau).

### Các hàm chính

| Hàm | Ai gọi | Kiểm tra | Log on-chain |
|---|---|---|---|
| `registerPatient(name, birthYear)` | Bệnh nhân | chưa đăng ký, không phải bác sĩ | `REGISTER_PATIENT` |
| `registerDoctor(name, specialty)` | Bác sĩ | chưa đăng ký, không phải bệnh nhân | event `DoctorRegistered` |
| `verifyDoctor(doctor)` | Bệnh viện | `msg.sender == hospital` | event `DoctorVerified` |
| `grantAccess(doctor)` | Bệnh nhân | bác sĩ đã được xác thực | `GRANT_ACCESS` |
| `revokeAccess(doctor)` | Bệnh nhân | bác sĩ đang có quyền | `REVOKE_ACCESS` |
| `addHealthRecord(patient, diagnosis, prescription, visitDate, note)` | Bệnh nhân hoặc bác sĩ có quyền ghi | `_canWrite` | `ADD_RECORD` |
| `viewHealthRecord(patient, index)` | Bệnh nhân hoặc bác sĩ có quyền đọc | `_canRead` | — (hàm `view`) |
| `logView(patient)` | Bác sĩ khi mở hồ sơ | `_canRead` | `VIEW_RECORD` |
| `getLog(patient, i)`, `getLogCount`, `getRecordCount`, `hasAccess` | Mọi người (log cần quyền đọc) | | |

**Điểm khác so với chữ ký trong đề:**

- `addHealthRecord` có thêm tham số `_patient` (bác sĩ cần chỉ rõ ghi vào hồ sơ của ai) và `_visitDate` (giao diện yêu cầu nhập ngày khám).
- `viewHealthRecord(patient, index)` trả về **1** hồ sơ theo chỉ số, vì Vyper không trả được mảng struct có độ dài thay đổi; giao diện lặp từ `0` đến `getRecordCount - 1`.
- Hàm `view` không ghi được dữ liệu, nên việc bác sĩ **xem** hồ sơ được ghi log bằng một giao dịch riêng `logView` (GUI tự gọi khi bác sĩ bấm "Xem hồ sơ").
- Thêm `verifyDoctor` để thể hiện vai trò Cơ sở y tế.


In [1]:
from pathlib import Path
print(Path(r"d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp\contracts\DHR.vy").read_text(encoding="utf-8"))


# @version ^0.4.3

"""
Digital Health Records DApp (BTTH3 - Bai tap ve nha).

Vai tro:
- Hospital (nguoi deploy): xac thuc bac si.
- Patient: tao ho so, cap / thu hoi quyen cho bac si, xem lich su.
- Doctor: dang ky, sau khi duoc xac thuc va duoc cap quyen thi xem / ghi ho so.
Moi thao tac deu ghi log on-chain (ActivityLog) va phat event.
"""

struct PatientRecord:
    name: String[64]
    birthYear: uint256
    recordCount: uint256
    logCount: uint256
    exists: bool

struct DoctorInfo:
    name: String[64]
    specialty: String[64]
    verified: bool
    exists: bool

struct AccessPermission:
    canRead: bool
    canWrite: bool
    grantedAt: uint256

struct HealthRecord:
    diagnosis: String[200]
    prescription: String[200]
    visitDate: String[32]
    note: String[200]
    doctor: address
    timestamp: uint256

struct ActivityLog:
    actor: address
    action: String[32]
    target: address
    timestamp: uint256

event PatientRegistered:
    patient: indexed(address)
   

## E. Biên dịch và unit test

```powershell
cd d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp
$ape = "d:\Study\IS355\.venv-ape\Scripts\ape.exe"
& $ape compile
& $ape test
```

Kết quả: `8 passed`.

| Test | Kiểm tra |
|---|---|
| `test_hospital` | Người deploy là `hospital` |
| `test_register` | Đăng ký 2 bệnh nhân + 1 bác sĩ, bác sĩ được xác thực |
| `test_register_twice_fail` | Đăng ký trùng bị từ chối |
| `test_verify_only_hospital` | Chỉ bệnh viện được xác thực bác sĩ |
| `test_grant_and_add_record` | Cấp quyền → bác sĩ ghi và đọc được hồ sơ |
| `test_doctor_without_access_fail` | Bác sĩ chưa được cấp quyền không ghi được |
| `test_revoke_access` | Thu hồi quyền → bác sĩ mất quyền ghi |
| `test_activity_log` | Log ghi đúng thứ tự `REGISTER_PATIENT → GRANT_ACCESS → ADD_RECORD → VIEW_RECORD` |

Kết quả `ape compile` và `ape test` nằm trong output của cell code ngay bên dưới.


In [2]:
import subprocess

APE = r"d:\Study\IS355\.venv-ape\Scripts\ape.exe"
PROJ = r"d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp"
NOISE = ("pydantic", "warnings.warn", "IncompleteField", "Danger!", "Using cached key")


def ape(*args):
    print(">", "ape", " ".join(args))
    r = subprocess.run([APE, *args], cwd=PROJ, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    for line in (r.stdout + r.stderr).splitlines():
        if line.strip() and not any(n in line for n in NOISE):
            print(line)
    print()


ape("compile")
ape("test")


> ape compile


SUCCESS:  'local project' compiled.

> ape test


============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-8.4.2, pluggy-1.6.0
rootdir: d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp
plugins: eth-ape-0.8.52, platformdirs-4.12.3
collected 8 items
tests\test_dhr.py ........                                               [100%]
============================== warnings summary ===============================
tests/test_dhr.py::test_hospital
-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 8 passed, 1 warning in 2.59s =========================



In [3]:
from pathlib import Path
base = Path(r"d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp\tests")
for name in ("conftest.py", "test_dhr.py"):
    print("=" * 60, name)
    print((base / name).read_text(encoding="utf-8"))


============================================================ conftest.py
import pytest


@pytest.fixture
def hospital(accounts):
    return accounts[0]


@pytest.fixture
def patient1(accounts):
    return accounts[1]


@pytest.fixture
def patient2(accounts):
    return accounts[2]


@pytest.fixture
def doctor(accounts):
    return accounts[3]


@pytest.fixture
def contract(hospital, project):
    return hospital.deploy(project.DHR)


@pytest.fixture
def setup(contract, hospital, patient1, patient2, doctor):
    contract.registerPatient("Nguyen Van A", 1990, sender=patient1)
    contract.registerPatient("Tran Thi B", 1995, sender=patient2)
    contract.registerDoctor("BS Le Van C", "Noi tong quat", sender=doctor)
    contract.verifyDoctor(doctor, sender=hospital)
    return contract

============================================================ test_dhr.py
import pytest
from ape.exceptions import ContractLogicError


def test_hospital(contract, hospital):
    assert contract.hospital() =

## F. Triển khai lên Geth cục bộ

**Terminal 1 — chạy Geth (giữ nguyên cửa sổ):**

```powershell
D:\Tools\Geth\geth.exe --dev --http --http.api eth,web3,net --http.addr 127.0.0.1 --http.port 8545 --ipcdisable
```

**Terminal 2 — tạo tài khoản, nạp ETH, deploy, chạy kịch bản kiểm thử:**

```powershell
cd d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp
$ape = "d:\Study\IS355\.venv-ape\Scripts\ape.exe"
$net = "ethereum:local:http://127.0.0.1:8545"

& $ape run create_accounts
& $ape run fund_accounts --network $net
& $ape run deploy --network $net
& $ape run demo_flow --network $net
```

- `create_accounts`: tạo 4 tài khoản Ape `hospital`, `patient1`, `patient2`, `doctor1` (mật khẩu `123456`); nếu đã có thì chỉ in địa chỉ.
- `fund_accounts`: chuyển 10 ETH từ dev account của Geth cho từng tài khoản (Geth `--dev` không lưu dữ liệu, mỗi lần bật lại phải nạp lại).
- `deploy`: `hospital` deploy `DHR`, địa chỉ contract được lưu vào `dhr_address.txt`.
- `demo_flow`: tạo 2 bệnh nhân + 1 bác sĩ, bệnh viện xác thực bác sĩ, `patient1` cấp quyền cho `doctor1`, bác sĩ ghi hồ sơ, `patient2` tự ghi hồ sơ, bác sĩ xem hồ sơ, rồi in hồ sơ + log; cuối cùng thử cho bác sĩ ghi vào hồ sơ `patient2` (chưa được cấp quyền) để xác nhận bị từ chối.

| Alias | Địa chỉ |
|---|---|
| hospital | `0xFc3dC6CccCB8981cf3229225B5968fdFF2A207c0` |
| patient1 | `0x0A0b7724887f3f20DAE226494226C1AbD2C38fd5` |
| patient2 | `0x3E71920b8ABa61F197d6F687A414E6E18a8Fa218` |
| doctor1 | `0xe0B40d4BfD55a40227DF95e6573ab86253f04c38` |

Kết quả chạy `create_accounts`, `fund_accounts`, `deploy` (có địa chỉ contract) và `demo_flow` nằm trong output của cell code bên dưới (sau phần in mã nguồn script).


In [4]:
from pathlib import Path
base = Path(r"d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp\scripts")
for name in ("create_accounts.py", "fund_accounts.py", "deploy.py", "demo_flow.py"):
    print("=" * 60, name)
    print((base / name).read_text(encoding="utf-8"))


============================================================ create_accounts.py
"""Tao 4 tai khoan Ape: hospital, patient1, patient2, doctor1 (password 123456).

Chay:
  ape run create_accounts
"""
from ape import accounts
from ape_accounts import generate_account

PASSWORD = "123456"
ALIASES = ("hospital", "patient1", "patient2", "doctor1")


def main():
    existing = set(accounts.aliases)
    for alias in ALIASES:
        if alias in existing:
            print(f"{alias:9} da ton tai: {accounts.load(alias).address}")
            continue
        account, _mnemonic = generate_account(alias, PASSWORD)
        print(f"{alias:9} tao moi : {account.address}")

============================================================ fund_accounts.py
"""Nap 10 ETH tu dev account cua Geth (eth.accounts[0]) cho 4 tai khoan DHR.

Chay:
  ape run fund_accounts --network ethereum:local:http://127.0.0.1:8545
"""
from ape import accounts, chain

ALIASES = ("hospital", "patient1", "patient2", "doctor1")


def

In [5]:
# Chay tren Geth dang bat: tao/kiem tra tai khoan, nap ETH, deploy, kich ban kiem thu
NET = "ethereum:local:http://127.0.0.1:8545"
ape("run", "create_accounts")
ape("run", "fund_accounts", "--network", NET)
ape("run", "deploy", "--network", NET)
ape("run", "demo_flow", "--network", NET)

> ape run create_accounts


hospital  da ton tai: 0xFc3dC6CccCB8981cf3229225B5968fdFF2A207c0
patient1  da ton tai: 0x0A0b7724887f3f20DAE226494226C1AbD2C38fd5
patient2  da ton tai: 0x3E71920b8ABa61F197d6F687A414E6E18a8Fa218
doctor1   da ton tai: 0xe0B40d4BfD55a40227DF95e6573ab86253f04c38

> ape run fund_accounts --network ethereum:local:http://127.0.0.1:8545


INFO:     Connecting to existing Geth node at http://127.0.0.1:8545.
Dev account Geth: 0x71562b71999873DB5b286dF957af199Ec94617F7
hospital  0xFc3dC6CccCB8981cf3229225B5968fdFF2A207c0  balance = 19.99913097348758811 ETH  (tx dd61f5564572fd18a4...)
patient1  0x0A0b7724887f3f20DAE226494226C1AbD2C38fd5  balance = 19.999147521143781083 ETH  (tx 26cbc9de3c912acfcd...)
patient2  0x3E71920b8ABa61F197d6F687A414E6E18a8Fa218  balance = 19.9990817494446608 ETH  (tx c37466e29297f564e7...)
doctor1   0xe0B40d4BfD55a40227DF95e6573ab86253f04c38  balance = 19.998374137628318315 ETH  (tx 0437bbf5ba92cacb58...)

> ape run deploy --network ethereum:local:http://127.0.0.1:8545


INFO:     Connecting to existing Geth node at http://127.0.0.1:8545.
INFO:     Contract will be deployed at: 0x7f2C30F0EfC8A6e89e2d57bBE66716b2029887Af
INFO:     Confirmed 0x719fe170498022bddf90538532a4f38ae4444c7cacaefda0308ede48e163c943 (total fees paid = 94043642577906)
SUCCESS:  Contract 'DHR' deployed to: 0x7f2C30F0EfC8A6e89e2d57bBE66716b2029887Af
DHR deployed at : 0x7f2C30F0EfC8A6e89e2d57bBE66716b2029887Af
Hospital        : 0xFc3dC6CccCB8981cf3229225B5968fdFF2A207c0
Da luu dia chi vao dhr_address.txt

> ape run demo_flow --network ethereum:local:http://127.0.0.1:8545


INFO:     Connecting to existing Geth node at http://127.0.0.1:8545.
DHR contract: 0x7f2C30F0EfC8A6e89e2d57bBE66716b2029887Af
INFO:     Confirmed 0x73f6093fd2fe6eee83e5687c293fd39370e82cf71586047ceeac5b51b1800849 (total fees paid = 66225197249802)
[patient1 registerPatient] tx=0x73f6093fd2fe6eee83e5687c293fd39370e82cf71586047ceeac5b51b1800849  status=confirmed  block=25
INFO:     Confirmed 0x83d65d42181eb93f459886c651eca5a3cea6373e23171c2776fcce25fb18e23e (total fees paid = 54974000580900)
[patient2 registerPatient] tx=0x83d65d42181eb93f459886c651eca5a3cea6373e23171c2776fcce25fb18e23e  status=confirmed  block=26
INFO:     Confirmed 0xfee9d6b9a6236fdb68f704ee6a44004fb93041771da6d10ecf6af514d52f7aa3 (total fees paid = 29857108635088)
[doctor1 registerDoctor] tx=0xfee9d6b9a6236fdb68f704ee6a44004fb93041771da6d10ecf6af514d52f7aa3  status=confirmed  block=27
INFO:     Confirmed 0x7c2108084dd84b07ca05bde2906d18c42882b01c32e8e1367c4bd4f1c3dc2e5b (total fees paid = 5295866328060)
[hospital veri

### Kết quả chạy `demo_flow` (log giao dịch)

Xem output của cell ngay phía trên (phần `ape run demo_flow`): mỗi giao dịch in ra tx hash, trạng thái và block; sau đó là hồ sơ và activity log của từng bệnh nhân.

**Giải thích:**

- Mỗi thao tác thay đổi trạng thái là 1 giao dịch được xác nhận trong 1 block riêng (Geth `--dev` đào block ngay khi có giao dịch).
- Log của `patient1` cho thấy đầy đủ ai làm gì: bệnh nhân tự đăng ký, cấp quyền cho `doctor1`, sau đó `doctor1` ghi và xem hồ sơ.
- `patient2` không cấp quyền cho bác sĩ nào, nên log chỉ có thao tác của chính bệnh nhân, và giao dịch ghi hồ sơ của `doctor1` vào `patient2` bị contract từ chối (`no write access`).


### Xem hình ảnh bên dưới (cuối file notebook)

## G. Giao diện GUI `dhr_gui_app.py`

Chạy (Geth đang chạy và đã deploy):

```powershell
cd d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp
d:\Study\IS355\.venv-ape\Scripts\python.exe dhr_gui_app.py
```

| Tab | Chức năng |
|---|---|
| **Đăng nhập** | Nhập RPC endpoint (`http://127.0.0.1:8545`), địa chỉ contract (tự đọc từ `dhr_address.txt`), địa chỉ ví + mật khẩu (có ô chọn nhanh tài khoản Ape). Sau khi đăng nhập hiển thị vai trò: Bệnh viện / Bệnh nhân / Bác sĩ (đã/chưa xác thực) / Chưa đăng ký và số dư. Có form đăng ký bệnh nhân, đăng ký bác sĩ, bệnh viện xác thực bác sĩ. |
| **Hồ sơ sức khỏe** | Nhập địa chỉ bệnh nhân → "Xem hồ sơ" hiển thị bảng các lần khám. Nếu người xem là bác sĩ, GUI gửi thêm giao dịch `logView` để ghi lại lượt xem. Form "Thêm hồ sơ mới": chẩn đoán, đơn thuốc, ngày khám, ghi chú. |
| **Cấp quyền truy cập** | Bệnh nhân nhập địa chỉ bác sĩ: cấp quyền đọc/ghi, thu hồi quyền, kiểm tra quyền hiện tại. |
| **Lịch sử truy cập** | Bảng log on-chain: thời gian, hành động, người thực hiện, đối tượng. |
| **Trạng thái giao dịch** | Mỗi giao dịch thêm 1 dòng: `pending` khi gửi, sau đó cập nhật transaction hash + `confirmed`/`failed` + block, và hiện hộp thoại thông báo khi hoàn tất. |

Giao dịch được gửi trên một luồng nền riêng (để giao diện không bị treo) và chạy **tuần tự**, tránh lỗi trùng nonce khi bác sĩ bấm "Xem hồ sơ" rồi "Thêm hồ sơ" liên tiếp.

### Kịch bản chụp ảnh cho báo cáo

1. Đăng nhập `patient1` (mật khẩu `123456`) → tab **Cấp quyền truy cập**: nhập địa chỉ `doctor1`, bấm "Kiểm tra quyền" rồi "Cấp quyền đọc/ghi" (nếu đã cấp từ `demo_flow` thì thử "Thu hồi quyền" rồi cấp lại).
2. Đóng GUI, mở lại, đăng nhập `doctor1` → tab **Hồ sơ sức khỏe**: nhập địa chỉ `patient1`, "Xem hồ sơ", rồi thêm 1 hồ sơ mới.
3. Tab **Lịch sử truy cập**: tải log của `patient1` → thấy `GRANT_ACCESS`, `ADD_RECORD`, `VIEW_RECORD` của `doctor1`.
4. Tab **Trạng thái giao dịch**: thấy các transaction hash đã `confirmed`.
5. (Tuỳ chọn) Đăng nhập `doctor1`, xem hồ sơ `patient2` → bị từ chối vì chưa được cấp quyền.

### Ảnh chụp giao diện

**Tab Đăng nhập (chưa đăng nhập):** RPC endpoint, địa chỉ contract tự đọc từ `dhr_address.txt`, ô chọn nhanh tài khoản.

![Đăng nhập](Images/g1_login.png)

**Đăng nhập `patient1`:** vai trò *Bệnh nhân* và số dư.

![Đăng nhập patient1](Images/g2_login_patient1.png)

**Đăng nhập `doctor1`:** vai trò *Bác sĩ (đã xác thực)*.

![Đăng nhập doctor1](Images/g3_login_doctor1.png)

**Tab Hồ sơ sức khỏe:** `doctor1` (đã được cấp quyền) xem hồ sơ của `patient1`.

![Bác sĩ xem hồ sơ](Images/g4_doctor_view_record.png)

**Tab Lịch sử truy cập:** log on-chain của `patient1` — `REGISTER_PATIENT`, `GRANT_ACCESS`, `ADD_RECORD`, `VIEW_RECORD` kèm địa chỉ ví và thời gian.

![Lịch sử truy cập](Images/g5_activity_log.png)

**Tab Trạng thái giao dịch:** giao dịch `logView` (ghi lượt xem của bác sĩ) đã `confirmed` kèm transaction hash và block.

![Trạng thái giao dịch](Images/g6_tx_status.png)

**Đăng nhập `patient2`:** bệnh nhân thứ hai.

![Đăng nhập patient2](Images/g7_login_patient2.png)

**Tab Hồ sơ sức khỏe — hộp thoại báo lỗi:** khi không đọc được hồ sơ, GUI hiện hộp thoại *Không xem được hồ sơ*; thanh trạng thái phía dưới cho thấy giao dịch `verifyDoctor` gửi trước đó đã `failed`.

![Hộp thoại lỗi](Images/g7b_records_error.png)

**`patient2` tự xem hồ sơ của mình:** 1 hồ sơ do chính bệnh nhân ghi (cột *Người ghi* là địa chỉ `patient2`). Bệnh nhân xem hồ sơ của mình nên không phát sinh giao dịch `logView`.

![patient2 xem hồ sơ](Images/g8_patient2_view_record.png)


In [6]:
from pathlib import Path
print(Path(r"d:\Study\IS355\TH\w3\BTTH3_VeNha\dhr_dapp\dhr_gui_app.py").read_text(encoding="utf-8"))


"""GUI PySide6 — Digital Health Records DApp (BTTH3 Ve Nha).

Chay (tu thu muc dhr_dapp, Geth dang chay, da deploy):
  d:\\Study\\IS355\\.venv-ape\\Scripts\\python.exe dhr_gui_app.py
"""
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

from ape import accounts, networks, project
from eth_utils import is_address
from PySide6 import QtCore, QtWidgets

DEFAULT_RPC = "http://127.0.0.1:8545"
ADDRESS_FILE = Path(__file__).resolve().parent / "dhr_address.txt"


def fmt_time(ts: int) -> str:
    return datetime.fromtimestamp(int(ts)).strftime("%Y-%m-%d %H:%M:%S")


def make_table(headers):
    table = QtWidgets.QTableWidget(0, len(headers))
    table.setHorizontalHeaderLabels(headers)
    table.horizontalHeader().setStretchLastSection(True)
    table.setEditTriggers(QtWidgets.QAbstractItemView.NoEditTriggers)
    return table


def fill_table(table, rows):
    table.setRowCount(0)
    for row in rows:
        r = table.rowCount()
        table.ins